# 10. 複素ベクトル空間と Jordan 標準形(付録)

| 層 | セクション |
|---|---|
| Basic | 1. Big Picture 〜 4. 複素固有値 |
| Applied | 5. 対角化できない行列 〜 6. Jordan 標準形 |
| Advanced | 7. 行列関数 / 8. Advanced Notes |

## 1. Big Picture

03 章で「回転は実固有ベクトルを持たない」「せん断は対角化できない」という
2 つの引っかかりを残しました。本章はその両方に決着をつけます。

- **複素ベクトル空間**: 複素数を許すと、回転にも固有値・固有ベクトルが現れる(複素固有値 = 回転)
- **Jordan 標準形**: 対角化できない行列も「対角 + わずかな上三角」まで必ず簡約できる

複素数を入れると **すべての $n \times n$ 行列が(複素数の範囲で)$n$ 個の固有値を持つ**
(代数学の基本定理)。線形代数の理論がここで完結します。

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

np.set_printoptions(precision=4, suppress=True)

## 2. 複素ベクトルと Hermitian 内積

複素ベクトル $\mathbb{C}^n$ では、内積に **共役** が入ります(長さが実数になるように)。

$$
\langle u, v \rangle = u^\dagger v = \sum_i \bar{u}_i\, v_i, \qquad \|v\|^2 = v^\dagger v \ge 0
$$

$u^\dagger = \bar{u}^\top$ は共役転置(随伴)。実ベクトルなら普通の内積に戻ります。
07 章の量子状態(Hermitian / Unitary 行列)はこの内積の上に乗っていました。

In [2]:
# Hermitian inner product: conjugate the first argument so norms stay real.
u = np.array([1 + 1j, 2 - 1j])
v = np.array([0 + 1j, 1 + 0j])
print("u^dagger v =", np.vdot(u, v))          # np.vdot conjugates the first arg
print("||u||^2 =", np.vdot(u, u).real, "(real and >= 0)")

u^dagger v = (3+2j)
||u||^2 = 7.0 (real and >= 0)


## 3. Unitary 行列 — 複素版の直交行列

$U^\dagger U = I$ を満たす行列が **Unitary**。実直交行列の複素版で、
**長さ(と Hermitian 内積)を保存** します(07 章の量子ゲート)。

In [3]:
# A unitary matrix preserves complex norms (here a phase gate times Hadamard).
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
phase = np.diag([1, 1j])
U = phase @ H
print("U^dagger U = I :", np.allclose(U.conj().T @ U, np.eye(2)))
x = np.array([1 + 0j, 0 + 0j])
print("||x|| =", np.linalg.norm(x), " ||U x|| =", np.linalg.norm(U @ x))

U^dagger U = I : True
||x|| = 1.0  ||U x|| = 0.9999999999999999


## 4. 複素固有値 = 回転

03 章の 90° 回転 $\begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ は実固有ベクトルを持ちませんでしたが、
複素数まで広げると固有値 $\pm i$ を持ちます。一般に **複素固有値 $re^{i\theta}$ の偏角 $\theta$ が回転、
絶対値 $r$ が拡大率** を表します。

In [4]:
# Complex eigenvalues encode rotation (angle) and scaling (magnitude).
theta = np.pi / 6
R = 1.2 * np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
w = np.linalg.eigvals(R)
print("eigenvalues:", w)
print("magnitudes (scaling):", np.abs(w), " -> matches 1.2")
print("angles (rotation, deg):", np.degrees(np.angle(w)), " -> matches +-30")

eigenvalues: [1.0392+0.6j 1.0392-0.6j]
magnitudes (scaling): [1.2 1.2]  -> matches 1.2
angles (rotation, deg): [ 30. -30.]  -> matches +-30


## 5. 対角化できない行列(復習と一般化)

03 章のせん断 $\begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}$ は固有値 1(重複)に対し
固有ベクトルが 1 本しかなく、対角化不能(**defective**)でした。
重複固有値の「代数的重複度」と「幾何的重複度」が食い違うのが原因です。

In [5]:
# Defective: algebraic multiplicity 2, but only 1 independent eigenvector.
S = np.array([[1.0, 1.0], [0.0, 1.0]])
w, V = np.linalg.eig(S)
print("eigenvalues:", w)
print("eigenvectors (columns nearly parallel -> only 1 direction):\n", V)
print("rank of eigenvector matrix:", np.linalg.matrix_rank(V))

eigenvalues: [1. 1.]
eigenvectors (columns nearly parallel -> only 1 direction):
 [[ 1. -1.]
 [ 0.  0.]]
rank of eigenvector matrix: 1


## 6. Jordan 標準形 — 対角化の一般化

対角化できない行列も、複素数の範囲で必ず **Jordan 標準形** に簡約できます:

$$
A = P J P^{-1}, \qquad J = \mathrm{diag}(J_1, \dots, J_m)
$$

各 **Jordan ブロック** $J_k$ は対角に固有値、その 1 つ上に 1 が並んだ形:

$$
J_k = \begin{pmatrix} \lambda & 1 & \\ & \lambda & \ddots \\ & & \ddots & 1 \\ & & & \lambda \end{pmatrix}
$$

対角化可能 = すべてのブロックが $1 \times 1$ の特別な場合。SymPy で計算します。

In [6]:
# Jordan form via SymPy: a defective 3x3 matrix becomes one 3x3 Jordan block.
A = sp.Matrix([[5, 4, 2], [0, 5, 1], [0, 0, 5]])
P, J = A.jordan_form()
print("Jordan form J =")
sp.pprint(J)
print("\nverify A = P J P^-1:", sp.simplify(P @ J @ P.inv() - A) == sp.zeros(3))

# The shear is itself a single 2x2 Jordan block.
_, Js = sp.Matrix([[1, 1], [0, 1]]).jordan_form()
print("\nshear Jordan form =", Js.tolist())

Jordan form J =
⎡5  1  0⎤
⎢       ⎥
⎢0  5  1⎥
⎢       ⎥
⎣0  0  5⎦

verify A = P J P^-1: True

shear Jordan form = [[1, 1], [0, 1]]


## 7. 行列関数 — Jordan / 対角化で計算する

対角化できる行列なら $f(A) = P\, f(\Lambda)\, P^{-1}$(各固有値に $f$ を適用)。
これで **行列指数関数** $e^{A}$ などが計算でき、07 章の量子時間発展
$e^{-iHt}$(Hermitian なら Unitary)に繋がります。

In [7]:
# Matrix exponential of a symmetric matrix via its eigendecomposition.
from scipy.linalg import expm

A = np.array([[2.0, 1.0], [1.0, 2.0]])
w, V = np.linalg.eigh(A)
expA_eig = V @ np.diag(np.exp(w)) @ V.T
print("max |expm(A) - V exp(Lambda) V^T| =", np.abs(expm(A) - expA_eig).max())

# e^{-iHt} of a Hermitian H is unitary (quantum time evolution, ch.07).
Hq = np.array([[0.0, 1.0], [1.0, 0.0]])      # Pauli X (Hermitian)
Ut = expm(-1j * Hq * 0.7)
print("e^{-iHt} unitary:", np.allclose(Ut.conj().T @ Ut, np.eye(2)))

max |expm(A) - V exp(Lambda) V^T| = 5.755396159656812e-13
e^{-iHt} unitary: True


## 8. まとめ

- **複素ベクトル空間**: Hermitian 内積 $u^\dagger v$、Unitary 行列が長さを保存(07 章量子の土台)。
- 複素数を許すと **すべての行列が $n$ 個の固有値** を持つ。複素固有値の偏角=回転・絶対値=拡大率。
- **Jordan 標準形**: 対角化できない行列も「対角 + 上の 1」まで必ず簡約できる。対角化はその特別な場合。
- **行列関数** $f(A)$ は対角化/Jordan で計算でき、$e^{-iHt}$(量子時間発展)に直結する。

## 9. Exercises

1. $\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$ の固有値が $e^{\pm i\theta}$ になることを確認せよ。
2. $\begin{pmatrix} 3 & 1 \\ 0 & 3 \end{pmatrix}$ の Jordan 形と、対角化できない理由を述べよ。
3. Pauli 行列 $X, Y, Z$ が Hermitian かつ Unitary であることを確認せよ。
4. $e^{A}$ を「対角化経由」と `scipy.linalg.expm` で計算し一致を確かめよ($A$ は対称)。
5. (発展)$3 \times 3$ で代数的重複度 3・幾何的重複度 2 の行列を作り、
   Jordan 形が $2 \times 2$ + $1 \times 1$ ブロックになることを確認せよ。